In [143]:
import pandas as pd
import re
import numpy as np

In [171]:
model_configs = [
    {'model': "Salesforce/blip2-flan-t5-xl", 'top_p': 'p06', 'temp': 't1', 'color': '#5ba3cf', 'accent_color': '#8cbedd'},
    {'model': "Salesforce/blip2-flan-t5-xxl", 'top_p': 'p1', 'temp': 't10', 'color': '#4c78a8', 'accent_color': '#81a0c2'},
    {'model': "llava-hf/llava-1.5-7b-hf", 'top_p': 'p04', 'temp': 't1', 'color': '#f9b574', 'accent_color': '#facb9d'},
    {'model': "llava-hf/llava-1.5-13b-hf", 'top_p': 'p1', 'temp': 't04', 'color': '#f58518', 'accent_color': '#f79d46'},
    {'model': "liuhaotian/llava-v1.6-34b", 'top_p': 'p1', 'temp': 't04', 'color': '#BF9000', 'accent_color': '#F1C232'},
    {'model': "google/pix2struct-chartqa-base", 'top_p': 'p08', 'temp': 't1', 'color': '#ba94c4', 'accent_color': '#c7b8d9'},
    {'model': "google/matcha-chartqa", 'top_p': 'p04', 'temp': 't1', 'color': '#8b6db2', 'accent_color': '#a28ac1'},
    {'model': "GPT-4V", 'top_p': 'p1', 'temp': 't02', 'color': '#c93739', 'accent_color': '#C6765E'},
    # {'model': "Human", 'top_p': 'pna', 'temp': 'tna', 'color': '#2e693b', 'accent_color': '#73D287'},
]

# tests = ['ggr', 'vlat', 'calvi', 'holf', 'holf2', 'chartqa-test-continuous-human']

## Model extraction procedure

In [169]:
questions = []
for test in ['ggr', 'vlat', 'calvi', 'holf', 'holf2', 'chartqa-test-continuous-human']:
    test_questions = pd.read_csv(f"https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/{test}/questions.csv")
    questions.append(test_questions)
questions = pd.concat(questions)

In [167]:
def extract_numerical_responses(word):
    """Extract all numbers from the response."""
    # remove all commas for example 1,000 -> 1000
    num_answer = str(word).replace(",", "")

    # find all negative numbers and decimals 
    pattern = r'-?\d+(\.\d+)?'
    all_numbers = list(re.finditer(pattern, num_answer))
    all_numbers = [float(num.group()) for num in all_numbers if num != '']

    return all_numbers

def find_non_floats(r):
    response_word_count = len(str(r['agent_response']).split(" "))
    
    # if only 2 or fewer words appear in the string
    if response_word_count <= 2:
        res = str(r['agent_response'])
        res = res.replace(" ", "")
        res = res.replace(",", "")
        extracted_response = extract_numerical_responses(res)
        if extracted_response:
            return extracted_response[0]
        else:
            return np.nan
    else:
        return np.nan

def find_non_mc(r):
    item = questions[
        (questions["question"] == r["question"]) &
        (questions["image_file"] == r["image_file"])                
    ].iloc[0]
    response = str(r["agent_response"]).lower()
    mc1, mc2, mc3, mc4 = item["mc1"], item["mc2"], item["mc3"], item["mc4"]

    for mc in [mc4,mc3,mc2,mc1]:
        if mc != np.nan and response.find(str(mc).lower().strip()) != -1:
            return mc

    return np.nan

def extract_valid_response(r):
    test_type = r['test_type']
    if test_type in ['holf', 'holf2', 'chartqa-test-continuous', 'chartqa-test-continuous-human']:
        return find_non_floats(r)
    elif test_type in ["vlat", 'calvi-trick', 'calvi-standard']:
        return find_non_mc(r)
    elif test_type in ['ggr']:
        item = questions[
            (questions["question"] == r["question"]) &
            (questions["image_file"] == r["image_file"])                
        ].iloc[0]

        if item['text_input_type'] == 'open_ended_question':
            return find_non_floats(r)
        elif item['text_input_type'] == 'multiple_choice_question':
            return find_non_mc(r)
        else:
            return np.nan

def extract_valid_response(r):
    test_type = r['test_type']
    if test_type in ['holf', 'holf2', 'chartqa-test-continuous', 'chartqa-test-continuous-human']:
        return find_non_floats(r)
    elif test_type in ["vlat", 'calvi-trick', 'calvi-standard']:
        return find_non_mc(r)
    elif test_type in ['ggr']:
        item = questions[
            (questions["question"] == r["question"]) &
            (questions["image_file"] == r["image_file"])                
        ].iloc[0]

        if item['text_input_type'] == 'open_ended_question':
            return find_non_floats(r)
        elif item['text_input_type'] == 'multiple_choice_question':
            return find_non_mc(r)
        else:
            return np.nan

### Process extracted model responses

In [173]:
model_responses = []
tests = ['chartqa-test-continuous-human']
aws_link = 'https://data-visualization-benchmark.s3.us-west-2.amazonaws.com'
for model_config in model_configs:
    model = model_config['model']
    top_p_dir = model_config['top_p']
    temperature_dir = model_config['temp']
    
    _dir = f'responses/indist_instructions_question/{top_p_dir}/{temperature_dir}/{model.replace("/", "-")}'
    for test_type in tests:
        file_name = f'{aws_link}/{test_type}/{_dir}/extracted_responses.csv'
        print(f"{file_name}")
        model_response_df = pd.read_csv(file_name)
        model_response_df['test_type'] = test_type
        model_responses.append(model_response_df)

model_responses = pd.concat(model_responses)
model_responses['item_id'] = model_responses['question'] + model_responses['image_file']
model_responses['model_response'] = model_responses.apply(extract_valid_response, axis=1)

https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/chartqa-test-continuous-human/responses/indist_instructions_question/p06/t1/Salesforce-blip2-flan-t5-xl/extracted_responses.csv
https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/chartqa-test-continuous-human/responses/indist_instructions_question/p1/t10/Salesforce-blip2-flan-t5-xxl/extracted_responses.csv
https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/chartqa-test-continuous-human/responses/indist_instructions_question/p04/t1/llava-hf-llava-1.5-7b-hf/extracted_responses.csv
https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/chartqa-test-continuous-human/responses/indist_instructions_question/p1/t04/llava-hf-llava-1.5-13b-hf/extracted_responses.csv
https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/chartqa-test-continuous-human/responses/indist_instructions_question/p1/t04/liuhaotian-llava-v1.6-34b/extracted_responses.csv
https://data-visualization-benchmark.s3.us-wes

## Mark model responses

In [176]:
def check_mc_answer(r):
    process_answer = lambda a : str(a).lower().strip()
    if (r['test_type'] in ['ggr', 'vlat']) and r['agent_type'] == 'Human':
        return int(r['is_correct'])

    is_correct = process_answer(r['correct_answer']) == process_answer(r['agent_response'])

    # there may be multiple correct answers so check if this  is the case
    is_correct_2 = False
    is_correct_3 = False
    try:
        is_correct_2 = process_answer(r['correct_answer_2']) == process_answer(r['agent_response'])
        is_correct_3 = process_answer(r['correct_answer_3']) == process_answer(r['agent_response'])
    except:
        pass

    return int(is_correct or is_correct_2 or is_correct_3)

def find_correct_mc(model_responses_raw):
    """
    Find if the model response has the correct multiple choice answer
    """
    model_responses = model_responses_raw.copy()
    model_responses["is_correct"] = model_responses.apply(
        check_mc_answer, axis=1
    )

    return model_responses

In [177]:
def get_relaxed_accuracy(agent_response, correct_answer, relax_accuracy_alpha):
    actual = abs(agent_response)
    correct = abs(correct_answer)
    return int((actual - correct) <= (correct * relax_accuracy_alpha))

def find_relaxed_accuracy(model_responses_raw, test_name, questions, relax_acc_alpha=0, dropna=True):
    """
    Find relaxed accuracy
    """
    model_responses = model_responses_raw.copy()
    model_responses["agent_response"] = model_responses["agent_response"].astype(float)  
    model_responses["is_correct"] = model_responses.apply(
        lambda r: get_relaxed_accuracy(r['agent_response'], r['correct_answer'], relax_acc_alpha), axis=1
    )

    return model_responses

In [150]:
def prepare_response_dataframe(
    model_responses, 
    test_type, 
    relax_acc=False, 
    relax_acc_alpha=0, 
    dropna=True
):
    if test_type in ['ggr', 'vlat', 'calvi-trick', 'calvi-standard']:
        model_responses = find_correct_mc(model_responses_raw=model_responses)
    else:
        if not relax_acc:
            model_responses = find_min_max_scaled_error(
                model_responses_raw=model_responses, 
                test_name=test_type,
                units_of_measure="is_correct",
                dropna=dropna
            )
        else:
            model_responses = find_relaxed_accuracy(
                model_responses_raw=model_responses, 
                test_name=test_type,
                units_of_measure="is_correct",
                relax_acc_alpha = relax_acc_alpha,
                dropna=dropna
            )

In [66]:
# model_responses

In [174]:
item_count = model_responses[['agentType', 'item_id']].value_counts().reset_index()
print(item_count['count'].unique())
item_count

[10  1]


,agentType,item_id,count
0,GPT-4V,"Find missing value- 2.9, 2.9, 3.5, 4.5, 5.6, 6...",10
1,google/pix2struct-chartqa-base,What was the fertility rate in Thailand in 201...,10
2,google/pix2struct-chartqa-base,What percentage of male students achieved a C/...,10
3,google/pix2struct-chartqa-base,What percentage of people were entering treatm...,10
4,google/pix2struct-chartqa-base,What percentage was the travel and tourism ind...,10
...,...,...,...
2635,llava-hf/llava-1.5-13b-hf,What is the Quarterly average in Q4 for the ye...,1
2636,llava-hf/llava-1.5-13b-hf,What is the % who regularly watch CNN in 02?15...,1
2637,llava-hf/llava-1.5-13b-hf,What is sum of the dark blue bar from 2019 to ...,1
2638,llava-hf/llava-1.5-13b-hf,What is sum of highest and lowest?two_col_2230...,1


In [157]:
# item_count[item_count['count'] == 1]

In [158]:
# item_count[['agentType']].value_counts()

In [159]:
# model_responses

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,_id,question,prompt,agentType,testType,correctAnswer,agent_response,...,topP,correct_answer,relaxed_accuracy,relaxed_accuracy_e0,normalize_by_correct_answer,absolute_error,agent_response.1,item_id,test_type,model_response
0,0,2630247,2630247,6650323b495a95861b3b8993,What was the total length of the gas pipeline ...,Question: You will be presented with a series ...,Salesforce/blip2-flan-t5-xl,chartqa-test-continuous,2582603.00,74080 miles,...,0.6,2582603.00,0,0,0.971316,2508523.0,74080 miles,What was the total length of the gas pipeline ...,chartqa-test-continuous,74080.00
1,1,2630249,2630249,6650323c495a95861b3b8997,What was the total length of the gas pipeline ...,Question: You will be presented with a series ...,Salesforce/blip2-flan-t5-xl,chartqa-test-continuous,2582603.00,3,...,0.6,2582603.00,0,0,0.999999,2582600.0,3,What was the total length of the gas pipeline ...,chartqa-test-continuous,3.00
2,2,2630254,2630254,6650323d495a95861b3b89a1,What was the total length of the gas pipeline ...,Question: You will be presented with a series ...,Salesforce/blip2-flan-t5-xl,chartqa-test-continuous,2582603.00,67518 miles,...,0.6,2582603.00,0,0,0.973857,2515085.0,67518 miles,What was the total length of the gas pipeline ...,chartqa-test-continuous,67518.00
3,3,2628563,2628563,66503068495a95861b3b7c6b,What was the total length of the gas pipeline ...,Question: You will be presented with a series ...,Salesforce/blip2-flan-t5-xl,chartqa-test-continuous,2582603.00,61038 mi,...,0.6,2582603.00,0,0,0.976366,2521565.0,61038 mi,What was the total length of the gas pipeline ...,chartqa-test-continuous,61038.00
4,4,2628567,2628567,66503069495a95861b3b7c73,What was the total length of the gas pipeline ...,Question: You will be presented with a series ...,Salesforce/blip2-flan-t5-xl,chartqa-test-continuous,2582603.00,58814 miles,...,0.6,2582603.00,0,0,0.977227,2523789.0,58814 miles,What was the total length of the gas pipeline ...,chartqa-test-continuous,58814.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1245,1245,80500,80500,6650b76b495a95861b3cbec1,What is the average of all produces less than ...,Question: You will be presented with a series ...,GPT-4V,chartqa-test-continuous,7.76,7.76,...,1.0,7.76,1,1,0.000000,0.0,7.76,What is the average of all produces less than ...,chartqa-test-continuous,7.76
1246,1246,80501,80501,6650b77e495a95861b3cbec3,What is the average of all produces less than ...,Question: You will be presented with a series ...,GPT-4V,chartqa-test-continuous,7.76,7.76,...,1.0,7.76,1,1,0.000000,0.0,7.76,What is the average of all produces less than ...,chartqa-test-continuous,7.76
1247,1247,80502,80502,6650b78e495a95861b3cbec5,What is the average of all produces less than ...,Question: You will be presented with a series ...,GPT-4V,chartqa-test-continuous,7.76,7.76,...,1.0,7.76,1,1,0.000000,0.0,7.76,What is the average of all produces less than ...,chartqa-test-continuous,7.76
1248,1248,72680,72680,664e9459495a95861b3a304d,What is the average of all produces less than ...,Question: You will be presented with a series ...,GPT-4V,chartqa-test-continuous,7.76,7.76,...,1.0,7.76,1,1,0.000000,0.0,7.76,What is the average of all produces less than ...,chartqa-test-continuous,7.76


In [154]:
# cqa_responses = prepare_response_dataframe("chartqa-test-continuous-human", relax_acc=True, relax_acc_alpha=0.0, dropna=False)

In [155]:
cqa_responses['item_id'] = cqa_responses['question'] + cqa_responses['image_file']
cqa_responses2 = cqa_responses[cqa_responses['agent_type'] != 'Human']
cqa_responses2[['agent_type', 'item_id']].value_counts().reset_index().sort_values(by='count')

,agent_type,item_id,count
0,GPT-4V,"Find missing value- 2.9, 2.9, 3.5, 4.5, 5.6, 6...",2
658,GPT-4V,How many units of AR glasses are expected to b...,2
659,GPT-4V,How many tourists arrived at accommodation est...,2
660,GPT-4V,How many 'Very' bars have a value lower than 4...,2
661,GPT-4V,How many Fairly well bars repeat themselves?19...,2
...,...,...,...
339,llava-hf/llava-1.5-7b-hf,what is the average of all No confidence data?...,2
340,llava-hf/llava-1.5-7b-hf,what is the average value of last three years ...,2
341,llava-hf/llava-1.5-7b-hf,what is the difference between retail and whol...,2
328,llava-hf/llava-1.5-7b-hf,What's the Age structure from 2009 fpr 0-14?mu...,2
